# Efficient LLM inference, by the numbers: KV cache, quantization, MoE, batching, speculative decoding

An LLM service wants the **same tokens, faster and cheaper**. This notebook is the hands-on side of that idea. Every number below is computed in a cell, so you can change an input and watch the budget move.

| Part | What you do | Changes the outputs? |
|---|---|---|
| 1 | Memory and speed budget: weights, KV cache, the decode speed limit, dollars | (arithmetic) |
| 2 | A tiny decoder with and without a KV cache; chunked prefill; prefix reuse; GQA | cache: no. GQA: yes |
| 3 | Absmax quantization: error vs bits; pruning; distillation temperature | yes, slightly or more |
| 4 | Mixture-of-experts: top-$k$ routing and expert load | yes (a different model) |
| 5 | Static vs continuous batching: count the idle slots | no |
| 6 | Speculative decoding: the accept/reject rule, checked empirically | no (by design) |

**How it works.** There are six small `TODO` functions. Each is followed by a check cell. Until you fill a `TODO`, its check prints a reminder and skips, so the notebook always runs top to bottom.

Everything runs on a CPU in a few minutes. All models have **random weights** and all data is synthetic, so nothing is downloaded. A random-weight model is enough here: a KV cache, a quantizer or a batch scheduler does not care whether the weights were trained. Timings depend on your machine, so read them as "on this run".

In [1]:
import math, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(4)
torch.manual_seed(0)
np.random.seed(0)
T_START = time.time()

def attempt(fn, *args, **kwargs):
    """Run a TODO function; if it is not written yet, say so and return None instead of crashing."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        print(f"[TODO] {fn.__name__} is not implemented yet: skipping this check.")
        return None

GB, MB, KB = 1e9, 1e6, 1e3
print("torch", torch.__version__)

torch 2.9.1


## Part 1. Where the cost goes: the budget

### One request, two phases

**Prefill** reads the whole prompt in one parallel pass and gives the first token. **Decode** then makes one token per step. So

$$\mathrm{latency} = \mathrm{TTFT} + \mathrm{TPOT}\times(\mathrm{output\ tokens}-1).$$

The slides' example: TTFT = 0.5 s, TPOT = 20 ms, a 200-token answer. The cell computes the latency and the decode speed for one user.

Prefill cost has two parts. Attention compares every token with every earlier one, so it grows like $n^2$. The MLP and the projections grow like $n$.

In [2]:
TTFT, TPOT, n_out = 0.5, 0.020, 200
latency = TTFT + TPOT * (n_out - 1)
print(f"latency = {TTFT} + {TPOT} x ({n_out} - 1) = {latency:.2f} s;  decode speed = {1/TPOT:.0f} tokens/s")
assert abs(latency - 4.48) < 1e-9 and round(1 / TPOT) == 50

for n in [1_000, 10_000]:
    print(f"prompt {n:>6,}: attention grows {(n/1000)**2:>5.0f}x,  MLP/projections grow {n/1000:>3.0f}x")

latency = 0.5 + 0.02 x (200 - 1) = 4.48 s;  decode speed = 50 tokens/s
prompt  1,000: attention grows     1x,  MLP/projections grow   1x
prompt 10,000: attention grows   100x,  MLP/projections grow  10x


### Memory: weights plus KV cache

Each layer stores a key and a value vector for every past token. With $L$ layers, model width $D$, $H$ query heads and $H_{kv}$ key/value heads (each head is $D/H$ wide):

$$\text{cache bytes per token} = 2\,L\,\frac{D}{H}\,H_{kv}\times\text{bytes per number}.$$

The factor 2 is for K and V. Ordinary multi-head attention has $H_{kv} = H$, so this is $2\,L\,D\times$bytes. Grouped-query attention (GQA) shares each K/V head across $H/H_{kv}$ query heads.

In [3]:
# TODO 1: KV-cache bytes for ONE token (all layers, K and V).
def kv_cache_bytes_per_token(n_layers, d_model, n_heads, n_kv_heads, bytes_per_value):
    """Return 2 * L * (D / H) * H_kv * bytes, as an int."""
    raise NotImplementedError

In [4]:
# Check 1: the slides' 7B example (L = 32, D = 4096, 32 heads, 16-bit) and Llama-2-70B (L = 80, D = 8192, 64 heads).
b7 = attempt(kv_cache_bytes_per_token, 32, 4096, 32, 32, 2)
if b7 is not None:
    print(f"7B, per token        : {b7:,} bytes = {b7/KB:.0f} KB")
    print(f"7B, 4,096 tokens     : {b7*4096/GB:.2f} GB per user;  8 users: {8*b7*4096/GB:.1f} GB")
    print(f"7B, 128,000 tokens   : {b7*128_000/GB:.1f} GB for one user ({128_000/4096:.2f}x the 4,096-token cache)")
    assert b7 == 524_288
    assert round(b7 * 4096 / GB, 1) == 2.1 and round(8 * b7 * 4096 / GB) == 17 and round(b7 * 128_000 / GB) == 67

    mha70 = kv_cache_bytes_per_token(80, 8192, 64, 64, 2)
    gqa70 = kv_cache_bytes_per_token(80, 8192, 64, 8, 2)
    print(f"Llama-2-70B, 64 KV heads: {mha70/MB:.1f} MB per token, {mha70*4096/GB:.1f} GB for 4,096 tokens")
    print(f"Llama-2-70B,  8 KV heads: {gqa70/KB:.0f} KB per token, {gqa70*4096/GB:.2f} GB for 4,096 tokens")
    assert round(mha70 / MB, 1) == 2.6 and round(gqa70 / KB) == 328 and mha70 // gqa70 == 8
    assert round(mha70 * 4096 / GB, 1) == 10.7 and round(gqa70 * 4096 / GB, 1) == 1.3

    # A check question from the slides: 16 heads -> 4 KV heads shrinks a 2.1 GB cache by 16/4 = 4 (not by 16).
    print(f"16 heads -> 4 KV heads: {b7*4096/GB:.2f} GB becomes {b7*4096/GB/4:.2f} GB")
    assert round(b7 * 4096 / GB / 4, 2) == 0.54
    print("Check 1 passed.")

[TODO] kv_cache_bytes_per_token is not implemented yet: skipping this check.


### The decode speed limit, and the price of a token

To make one token, decode reads (nearly) all the weights once. So for one request

$$\text{tokens/s} \lesssim \frac{\text{memory bandwidth}}{\text{model size in bytes}}.$$

It is an upper bound: it ignores the KV cache and the compute. The bandwidth (3 TB/s) and the GPU rent ($3 per hour, 100 tokens/s per request) are the slides' illustrative values, not measurements.

In [5]:
def decode_speed_limit(bandwidth_bytes_per_s, n_params, bits):
    return bandwidth_bytes_per_s / (n_params * bits / 8)

print("7B model at 3 TB/s:")
for bits, expect in [(16, 214), (8, 429), (4, 857)]:
    s = decode_speed_limit(3e12, 7e9, bits)
    print(f"  {bits:>2}-bit: {7e9*bits/8/GB:4.1f} GB -> at most {s:4.0f} tokens/s")
    assert round(s) == expect

# A check question: a 10B model at 2 TB/s.
for bits, expect in [(8, 200), (4, 400)]:
    assert round(decode_speed_limit(2e12, 10e9, bits)) == expect
print("10B model at 2 TB/s: 8-bit -> 200 tokens/s, 4-bit -> 400 tokens/s")

def cost_per_million(rent_per_hour, tokens_per_s_per_user, users):
    tokens_per_hour = tokens_per_s_per_user * 3600 * users
    return rent_per_hour / (tokens_per_hour / 1e6), tokens_per_hour

for users, expect in [(1, 8.33), (32, 0.26)]:
    c, tph = cost_per_million(3.0, 100, users)
    print(f"{users:>2} user(s): {tph/1e6:5.2f} M tokens/hour -> ${c:.2f} per million tokens")
    assert round(c, 2) == expect

7B model at 3 TB/s:
  16-bit: 14.0 GB -> at most  214 tokens/s
   8-bit:  7.0 GB -> at most  429 tokens/s
   4-bit:  3.5 GB -> at most  857 tokens/s
10B model at 2 TB/s: 8-bit -> 200 tokens/s, 4-bit -> 400 tokens/s
 1 user(s):  0.36 M tokens/hour -> $8.33 per million tokens
32 user(s): 11.52 M tokens/hour -> $0.26 per million tokens


### Stack the tricks: how many users fit on 80 GB?

Weights are shared by all users. Each user pays for a full 4,096-token cache. The cell counts users for the slides' four setups (memory only, 7B model, 80 GB GPU).

In [6]:
if b7 is not None:
    full, small = b7 * 4096, b7 * 4096 / 4            # GQA with a 4x smaller cache
    setups = [("16-bit weights, full cache", 16, full, 30),
              ("8-bit weights, full cache", 8, full, 33),
              ("8-bit weights, GQA 4x smaller", 8, small, 135),
              ("4-bit weights, GQA 4x smaller", 4, small, 142)]
    print(f"{'setup':32s} {'weights':>8s} {'cache/user':>11s} {'users':>6s}")
    for name, bits, cache, expect in setups:
        w = 7e9 * bits / 8
        users = int((80e9 - w) // cache)
        print(f"{name:32s} {w/GB:6.1f} GB {cache/GB:8.2f} GB {users:6d}")
        assert users == expect
    print("\nHalving the weights adds a few users; shrinking the cache adds a hundred. The cache sets capacity.")
else:
    print("[TODO] this table needs kv_cache_bytes_per_token: skipping.")

[TODO] this table needs kv_cache_bytes_per_token: skipping.


## Part 2. The KV cache in a tiny decoder

Below is a small decoder-only Transformer with **random weights** (pre-norm blocks, causal attention, an MLP). It is not trained and we will not train it. Its `forward` takes the **new** tokens plus an optional `cache`, a list with one `(K, V)` pair per layer. It appends the new keys and values to the cache and returns the logits of the new positions and the longer cache.

The constructor has `n_kv_heads`. With `n_kv_heads < n_heads` each K/V head is shared by a group of query heads: that is GQA.

In [7]:
class Attention(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads):
        super().__init__()
        self.H, self.H_kv, self.dh = n_heads, n_kv_heads, d // n_heads
        self.q = nn.Linear(d, d, bias=False)
        self.kv = nn.Linear(d, 2 * n_kv_heads * self.dh, bias=False)
        self.o = nn.Linear(d, d, bias=False)

    def forward(self, x, kv_past=None):
        B, n, _ = x.shape
        q = self.q(x).view(B, n, self.H, self.dh).transpose(1, 2)                  # (B, H, n, dh)
        k, v = self.kv(x).view(B, n, 2, self.H_kv, self.dh).permute(2, 0, 3, 1, 4)  # each (B, H_kv, n, dh)
        if kv_past is not None:                                   # the KV cache: old keys/values + new ones
            k = torch.cat([kv_past[0], k], dim=2)
            v = torch.cat([kv_past[1], v], dim=2)
        T = k.shape[2]
        rep = self.H // self.H_kv                                 # GQA: share each K/V head
        kk, vv = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        att = (q @ kk.transpose(-1, -2)) / math.sqrt(self.dh)     # (B, H, n, T)
        q_pos = torch.arange(T - n, T).view(n, 1)
        k_pos = torch.arange(T).view(1, T)
        att = att.masked_fill(k_pos > q_pos, float("-inf")).softmax(-1)
        y = (att @ vv).transpose(1, 2).reshape(B, n, -1)
        return self.o(y), (k, v)

class Block(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = Attention(d, n_heads, n_kv_heads)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x, kv_past=None):
        a, kv = self.attn(self.ln1(x), kv_past)
        x = x + a
        return x + self.mlp(self.ln2(x)), kv

class TinyDecoder(nn.Module):
    def __init__(self, vocab=512, d=256, n_layers=4, n_heads=8, n_kv_heads=None, max_len=1024):
        super().__init__()
        self.cfg = dict(vocab=vocab, d=d, n_layers=n_layers, n_heads=n_heads,
                        n_kv_heads=n_kv_heads or n_heads)
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d, n_heads, n_kv_heads or n_heads) for _ in range(n_layers)])
        self.ln_f, self.head = nn.LayerNorm(d), nn.Linear(d, vocab, bias=False)

    @torch.no_grad()
    def forward(self, idx, cache=None):
        """idx: (B, n) NEW tokens. cache: list of (K, V) per layer, or None.
        Returns logits (B, n, vocab) for the new positions, and the updated cache."""
        past = 0 if cache is None else cache[0][0].shape[2]
        x = self.tok(idx) + self.pos(torch.arange(past, past + idx.shape[1]))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, kv = blk(x, None if cache is None else cache[i])
            new_cache.append(kv)
        return self.head(self.ln_f(x)), new_cache

def cache_bytes(cache):
    return sum(t.numel() * t.element_size() for kv in cache for t in kv)

torch.manual_seed(0)
model = TinyDecoder().eval()
print(model.cfg, f"{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters")

{'vocab': 512, 'd': 256, 'n_layers': 4, 'n_heads': 8, 'n_kv_heads': 8} 3.68 M parameters


### Generation without a cache, and with one

Without a cache, every decode step runs the **whole** sequence again, so step $t$ costs about $t$ tokens of work. With a cache, prefill runs once over the prompt, and each decode step feeds **one** token plus the cache. The outputs must be the same: the cache only saves recomputation.

`generate_no_cache` is given. Write `generate_with_cache` (greedy decoding: pick the arg-max token).

In [8]:
def generate_no_cache(model, prompt, n_new):
    """Greedy decoding that recomputes the full sequence every step. Returns (tokens, list of logits rows)."""
    seq, rows = prompt.clone(), []
    for _ in range(n_new):
        logits, _ = model(seq)                 # whole sequence, no cache
        last = logits[:, -1]
        rows.append(last)
        seq = torch.cat([seq, last.argmax(-1, keepdim=True)], dim=1)
    return seq[:, prompt.shape[1]:], rows

In [9]:
# TODO 2: greedy decoding with a KV cache.
def generate_with_cache(model, prompt, n_new):
    """Return (tokens, rows) exactly like generate_no_cache, where tokens has shape (B, n_new)
    and rows is a list of the n_new last-position logits (each (B, vocab)).
    Steps: prefill: logits, cache = model(prompt). Then repeat n_new times:
    take the last logits row, append it to rows, pick next = argmax, and feed ONLY next
    (shape (B, 1)) together with the cache: logits, cache = model(next, cache)."""
    raise NotImplementedError

In [10]:
# Check 2: same tokens, same logits; then time both.
torch.manual_seed(1)
prompt = torch.randint(0, 512, (1, 64))
N_NEW = 256

t0 = time.perf_counter(); tok_a, rows_a = generate_no_cache(model, prompt, N_NEW); t_nocache = time.perf_counter() - t0
r = attempt(generate_with_cache, model, prompt, N_NEW)
if r is not None:
    t0 = time.perf_counter(); tok_b, rows_b = generate_with_cache(model, prompt, N_NEW); t_cache = time.perf_counter() - t0
    max_diff = max((a - b).abs().max().item() for a, b in zip(rows_a, rows_b))
    print(f"identical tokens: {torch.equal(tok_a, tok_b)};  largest logit difference: {max_diff:.1e}")
    assert tok_b.shape == (1, N_NEW) and torch.equal(tok_a, tok_b) and max_diff < 1e-3
    print(f"On this run: no cache {t_nocache:.2f} s, with cache {t_cache:.2f} s "
          f"-> {t_nocache/t_cache:.1f}x faster for {N_NEW} new tokens")
    print("Check 2 passed.")

[TODO] generate_with_cache is not implemented yet: skipping this check.


### TTFT, TPOT and the size of the cache

Now measure the two phases of one request on this tiny model, and compare the cache it actually stores with the formula from Part 1. The model uses 32-bit floats, so 4 bytes per number.

In [11]:
with torch.no_grad():
    t0 = time.perf_counter(); logits, cache = model(prompt); ttft = time.perf_counter() - t0
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    t0 = time.perf_counter()
    for _ in range(64):
        logits, cache = model(nxt, cache); nxt = logits[:, -1].argmax(-1, keepdim=True)
    tpot = (time.perf_counter() - t0) / 64
print(f"On this run: TTFT (prefill of {prompt.shape[1]} tokens) = {ttft*1e3:.1f} ms, TPOT = {tpot*1e3:.2f} ms")

T_now = cache[0][0].shape[2]
per_tok = cache_bytes(cache) / T_now
print(f"cache holds {T_now} tokens, {cache_bytes(cache):,} bytes -> {per_tok:,.0f} bytes per token")
if b7 is not None:
    c = model.cfg
    formula = kv_cache_bytes_per_token(c["n_layers"], c["d"], c["n_heads"], c["n_kv_heads"], 4)
    print(f"formula 2 L D x 4 bytes = {formula:,} bytes per token")
    assert per_tok == formula

On this run: TTFT (prefill of 64 tokens) = 2.3 ms, TPOT = 0.50 ms
cache holds 128 tokens, 1,048,576 bytes -> 8,192 bytes per token


### Two exact serving tricks: chunked prefill and prefix reuse

Both work because the cache after a prefix depends only on that prefix.

* **Chunked prefill** (Sarathi-Serve): prefill a long prompt in small chunks, so that decodes of other users can run between chunks. The resulting cache is the same.
* **Prefix caching** (SGLang's RadixAttention, vLLM): many requests start with the same system prompt. Prefill it once, keep its cache, and prefill only each user's new part.

The cell checks that both give the same logits as one full prefill.

In [12]:
torch.manual_seed(2)
system = torch.randint(0, 512, (1, 96))                       # shared system prompt + tools
users = [torch.randint(0, 512, (1, 24)) for _ in range(3)]    # each user's new turn

# Chunked prefill of the system prompt, 16 tokens at a time.
full_logits, full_cache = model(system)
cache = None
for s in range(0, system.shape[1], 16):
    chunk_logits, cache = model(system[:, s:s + 16], cache)
d_chunk = (chunk_logits[:, -1] - full_logits[:, -1]).abs().max().item()
d_cache = max((a - b).abs().max().item() for kv1, kv2 in zip(cache, full_cache) for a, b in zip(kv1, kv2))
print(f"chunked prefill: last-logit difference {d_chunk:.1e}, cache difference {d_cache:.1e}")
assert d_chunk < 1e-4 and d_cache < 1e-4

# Prefix reuse: prefill the system prompt once, then only each user's turn.
saved_tokens = 0
for u in users:
    ref, _ = model(torch.cat([system, u], dim=1))             # full prefill
    reuse, _ = model(u, full_cache)                           # cache hit on the prefix
    d = (ref[:, -u.shape[1]:] - reuse).abs().max().item()
    assert d < 1e-4
    saved_tokens += system.shape[1]
total = len(users) * (system.shape[1] + users[0].shape[1])
print(f"prefix reuse: same logits for all {len(users)} users; prefilled {total - saved_tokens + system.shape[1]} "
      f"tokens instead of {total}")

chunked prefill: last-logit difference 0.0e+00, cache difference 0.0e+00
prefix reuse: same logits for all 3 users; prefilled 168 tokens instead of 360


### GQA: fewer K/V heads, a smaller cache

Build the same decoder with 8 query heads but only 2 K/V heads. The cache should shrink by $H / H_{kv} = 4$. Note that this is a **different model**: unlike the cache itself, GQA changes the outputs, so a GQA model must be trained (or converted and fine-tuned) and then evaluated.

In [13]:
torch.manual_seed(0)
model_gqa = TinyDecoder(n_kv_heads=2).eval()
_, c_mha = model(prompt)
_, c_gqa = model_gqa(prompt)
ratio = cache_bytes(c_mha) / cache_bytes(c_gqa)
print(f"cache for {prompt.shape[1]} tokens: 8 KV heads {cache_bytes(c_mha):,} bytes, "
      f"2 KV heads {cache_bytes(c_gqa):,} bytes -> {ratio:.0f}x smaller")
assert ratio == 4
if b7 is not None:
    assert cache_bytes(c_gqa) == prompt.shape[1] * kv_cache_bytes_per_token(4, 256, 8, 2, 4)

cache for 64 tokens: 8 KV heads 524,288 bytes, 2 KV heads 131,072 bytes -> 4x smaller


## Part 3. Smaller models: quantization, pruning, distillation

### Absmax quantization, one scale per block

Store each weight as a small integer and keep one floating-point **scale** per block of weights. With $b$ bits the integers run from $-q_{\max}$ to $q_{\max}$, where $q_{\max} = 2^{b-1} - 1$ (127 for 8 bits, 7 for 4 bits). For each block:

$$s = \frac{\max_i |w_i|}{q_{\max}}, \qquad q_i = \mathrm{round}(w_i / s), \qquad \hat w_i = s\, q_i .$$

A small block keeps one large weight (an **outlier**) from forcing a coarse step on all the others. The scale costs extra memory: with a 16-bit scale per block of size $B$, the effective bits per weight are $b + 16/B$.

In [14]:
# TODO 3: absmax quantization with one scale per block.
def quantize_absmax(w, bits, block_size):
    """w: float tensor whose numel is divisible by block_size.
    Return (q, scale): q is an int8 tensor of shape (n_blocks, block_size) with values in [-qmax, qmax],
    scale is a float tensor of shape (n_blocks, 1). Use qmax = 2**(bits-1) - 1.
    Hint: reshape w to (-1, block_size); guard against an all-zero block with .clamp(min=1e-12)."""
    raise NotImplementedError

In [15]:
def dequantize(q, scale, shape):
    return (q.float() * scale).reshape(shape)

def rel_error(w, w_hat):
    return ((w - w_hat).norm() / w.norm()).item()

# A weight matrix with a few large outliers, as seen in real LLM weights and activations.
torch.manual_seed(3)
W = torch.randn(1024, 1024) * 0.02
W.view(-1)[torch.randperm(W.numel())[:20]] = 0.5

r = attempt(quantize_absmax, W, 8, 64)
if r is not None:
    q, s = r
    assert q.dtype == torch.int8 and q.shape == (W.numel() // 64, 64) and s.shape == (W.numel() // 64, 1)
    assert q.abs().max() <= 127
    W8 = dequantize(q, s, W.shape)
    assert ((W - W8).abs().reshape(-1, 64) <= s / 2 + 1e-7).all()   # rounding error is at most half a step

    rows, errs = [], {}
    for bits in [8, 6, 4, 3, 2]:
        for block in [W.numel(), 64]:                 # one scale for the whole tensor, vs one per 64 weights
            q, s = quantize_absmax(W, bits, block)
            e = rel_error(W, dequantize(q, s, W.shape))
            errs[(bits, block)] = e
            label = "whole tensor" if block == W.numel() else f"block {block}"
            rows.append((bits, label, bits + 16 / block, e))
    print(f"{'bits':>4s}  {'scale per':14s} {'effective bits':>14s} {'relative error':>15s}")
    for bits, label, eff, e in rows:
        print(f"{bits:>4d}  {label:14s} {eff:14.2f} {e:15.4f}")
    for block in [W.numel(), 64]:
        es = [errs[(b, block)] for b in [8, 6, 4, 3, 2]]
        assert all(a < b for a, b in zip(es, es[1:]))            # fewer bits, more error
    assert all(errs[(b, 64)] < errs[(b, W.numel())] for b in [8, 6, 4, 3])   # blocks tame outliers
    assert errs[(8, 64)] < 0.01
    per_bit = [(errs[(b2, 64)] / errs[(b1, 64)]) ** (1 / (b1 - b2)) for b1, b2 in [(8, 6), (6, 4), (4, 3), (3, 2)]]
    print("error growth per bit removed (block 64):", ", ".join(f"{g:.1f}x" for g in per_bit))
    assert all(1.5 < g < 3.0 for g in per_bit)                   # roughly doubles per bit
    assert errs[(3, W.numel())] > 0.95                           # one scale: ordinary weights round to zero
    print("Check 3 passed.")

[TODO] quantize_absmax is not implemented yet: skipping this check.


Read the table. With per-block scales, each bit removed roughly doubles the error. With one scale for the whole tensor, the 20 outliers set the step size, so at 3 bits and below almost every ordinary weight rounds to zero (relative error near 1). Per-block scales cost only 0.25 extra bits per weight at block size 64.

### Does quantization change the model's outputs?

Quantize every linear layer of the tiny decoder (block size 64) and compare its next-token logits with the original on 64 random prompts. Quantization is a **model-level** trick: the outputs move, so it needs an evaluation.

In [16]:
def quantize_model(m, bits, block=64):
    mq = copy.deepcopy(m)
    for mod in mq.modules():
        if isinstance(mod, nn.Linear):
            q, s = quantize_absmax(mod.weight.data, bits, block)
            mod.weight.data = dequantize(q, s, mod.weight.shape)
    return mq

if r is not None:
    torch.manual_seed(4)
    prompts = torch.randint(0, 512, (64, 32))
    ref, _ = model(prompts)
    ref = ref[:, -1]
    for bits in [8, 4, 3]:
        out, _ = quantize_model(model, bits)(prompts)
        out = out[:, -1]
        agree = (out.argmax(-1) == ref.argmax(-1)).float().mean().item()
        print(f"{bits}-bit weights: logit relative error {rel_error(ref, out):.4f}, "
              f"same top-1 token on {agree:.0%} of prompts")
else:
    print("[TODO] this comparison needs quantize_absmax: skipping.")

[TODO] this comparison needs quantize_absmax: skipping.


### Pruning: zeros are not free

Unstructured pruning sets individual weights to zero, but the matrix keeps its shape, so a dense matrix multiply does the same work. Structured pruning removes whole rows, which really shrinks the multiply. Compare the times on this run.

In [17]:
torch.manual_seed(5)
Wd = torch.randn(2048, 2048)
X = torch.randn(2048, 256)
W_unstructured = Wd * (torch.rand_like(Wd) > 0.5)     # half the weights zeroed, same shape
W_structured = Wd[:1024]                              # half the rows removed

def bench(A, reps=20):
    A @ X
    t0 = time.perf_counter()
    for _ in range(reps):
        A @ X
    return (time.perf_counter() - t0) / reps

t_dense, t_unstr, t_str = bench(Wd), bench(W_unstructured), bench(W_structured)
print(f"zeros in unstructured matrix: {(W_unstructured == 0).float().mean():.0%}")
print(f"On this run: dense {t_dense*1e3:.2f} ms, 50% unstructured {t_unstr*1e3:.2f} ms, "
      f"50% structured {t_str*1e3:.2f} ms")

zeros in unstructured matrix: 50%
On this run: dense 1.25 ms, 50% unstructured 1.20 ms, 50% structured 0.63 ms


### Distillation: soft targets

A student learns from the teacher's softened probabilities $p_i = \exp(z_i/T) / \sum_j \exp(z_j/T)$. The slides' teacher logits are 4, 1, 0. At $T = 1$ the targets are almost a hard label; at $T = 4$ the student can see that class 2 beats class 3; as $T$ grows the targets approach uniform and the signal fades.

In [18]:
z = torch.tensor([4.0, 1.0, 0.0])
for T in [1, 4, 100]:
    p = (z / T).softmax(-1)
    print(f"T = {T:>3}: " + ", ".join(f"{v:.2f}" for v in p.tolist()))
assert [round(v, 2) for v in (z / 1).softmax(-1).tolist()] == [0.94, 0.05, 0.02]
assert [round(v, 2) for v in (z / 4).softmax(-1).tolist()] == [0.54, 0.26, 0.20]

T =   1: 0.94, 0.05, 0.02
T =   4: 0.54, 0.26, 0.20
T = 100: 0.34, 0.33, 0.33


## Part 4. Mixture-of-experts: compute only what you need

An MoE layer replaces one feed-forward block by $E$ experts and a small **router**. For each token the router gives $E$ scores; the token goes to its top-$k$ experts, and their outputs are mixed with gates that are a softmax over the $k$ chosen scores (Shazeer et al. 2017 keep the top $k$ and softmax over them; Switch Transformers use $k = 1$).

In [19]:
# TODO 4: top-k routing.
def top_k_route(router_logits, k):
    """router_logits: (N, E). Return (idx, gates), both (N, k):
    idx are the k highest-scoring experts per token, gates = softmax over those k scores (each row sums to 1).
    Hint: torch.topk."""
    raise NotImplementedError

In [20]:
E, K, d_moe = 16, 2, 64
torch.manual_seed(6)
router = nn.Linear(d_moe, E, bias=False)
tokens = torch.randn(4096, d_moe)
with torch.no_grad():
    logits_r = router(tokens)

r = attempt(top_k_route, logits_r, K)
if r is not None:
    idx, gates = r
    assert idx.shape == (4096, K) and gates.shape == (4096, K)
    assert torch.allclose(gates.sum(-1), torch.ones(4096), atol=1e-6)
    assert torch.equal(top_k_route(logits_r, 1)[0][:, 0], logits_r.argmax(-1))
    chosen = torch.gather(logits_r, 1, idx)
    others = logits_r.masked_fill(torch.zeros_like(logits_r, dtype=torch.bool).scatter(1, idx, True), -1e9)
    assert (chosen.min(-1).values >= others.max(-1).values).all()     # chose the best k

    load = torch.bincount(idx.flatten(), minlength=E)
    print("tokens per expert:", load.tolist())
    print(f"busiest expert gets {load.max().item() / load.float().mean().item():.2f}x the average load")

    # Switch Transformer load-balancing loss: E * sum_i f_i * P_i (equals 1 when perfectly balanced).
    f = torch.bincount(logits_r.argmax(-1), minlength=E).float() / len(tokens)
    P = logits_r.softmax(-1).mean(0)
    print(f"load-balancing loss E * sum f_i P_i = {(E * (f * P).sum()).item():.3f}  (1.0 = perfectly balanced)")
    print("Check 4 passed.")

[TODO] top_k_route is not implemented yet: skipping this check.


### Active parameters vs memory, and why big batches touch every expert

Each token uses only $k$ of $E$ experts, so the compute per token is a fraction of the stored parameters. But a batch spreads its tokens across experts. If routing were uniform, a batch of $N$ tokens would touch a fraction $1 - (1 - k/E)^N$ of the experts. The cell measures this with our router. Then it applies the same share arithmetic to the slides' large-model example (1.6 T total, 49 B active per token).

In [21]:
if r is not None:
    d_ff = 4 * d_moe
    expert_params = 2 * d_moe * d_ff
    print(f"{E} experts x {expert_params:,} params = {E*expert_params:,} stored; "
          f"{K} x {expert_params:,} = {K*expert_params:,} used per token ({K/E:.1%})")

    g = torch.Generator().manual_seed(7)
    print(f"{'batch':>6s} {'experts touched (measured)':>27s} {'uniform formula':>16s}")
    touched = {}
    for N in [1, 4, 16, 64]:
        fr = []
        for _ in range(200):
            sel = torch.randint(0, len(tokens), (N,), generator=g)
            fr.append(idx[sel].unique().numel() / E)
        touched[N] = sum(fr) / len(fr)
        print(f"{N:>6d} {touched[N]:>27.0%} {1 - (1 - K/E)**N:>16.0%}")
    assert touched[1] == K / E and touched[64] > 0.9

    print(f"slides' example: 49 B active of 1.6 T total = {49/1600:.1%} of the weights per token, "
          f"but all {1.6e12*2/1e12:.1f} TB (at 16 bits) must stay in memory")
else:
    print("[TODO] this cell needs top_k_route: skipping.")

[TODO] this cell needs top_k_route: skipping.


## Part 5. Batching: count the idle slots

A GPU runs a batch of `n_slots` requests together, one token per request per step.

* **Static batching** takes requests in arrival order, `n_slots` at a time, and runs each batch until its **longest** request is done. Short requests leave their slot idle.
* **Continuous batching** (Orca's iteration-level scheduling) refills a free slot from the queue at the next step.

Utilization = useful tokens / slot-steps. The slides' example has four requests needing 2, 8, 3 and 5 tokens on 4 slots.

In [22]:
# TODO 5: static batching.
def static_batching(lengths, n_slots):
    """lengths: tokens each request needs, in arrival order. Return (slot_steps, used, utilization).
    Batches are lengths[0:n_slots], lengths[n_slots:2*n_slots], ...; each batch runs max(batch) steps
    and occupies n_slots slots for all of them. used = sum(lengths)."""
    raise NotImplementedError

In [23]:
def continuous_batching(lengths, n_slots):
    """Each step, every free slot takes the next waiting request. Returns (slot_steps, used, utilization)."""
    queue, slots, steps = list(lengths), [0] * n_slots, 0
    while queue or any(slots):
        for i in range(n_slots):
            if slots[i] == 0 and queue:
                slots[i] = queue.pop(0)
        slots = [max(s - 1, 0) for s in slots]
        steps += 1
    return n_slots * steps, sum(lengths), sum(lengths) / (n_slots * steps)

r = attempt(static_batching, [2, 8, 3, 5], 4)
if r is not None:
    print(f"slides' example, static: {r[0]} slot-steps, {r[1]} used -> {r[2]:.0%}")
    assert r == (32, 18, 18 / 32)
    rc = continuous_batching([2, 8, 3, 5], 4)
    print(f"slides' example, continuous with no queue: {rc[2]:.0%} (the same: nothing is waiting)")

    rng = np.random.default_rng(8)
    lengths = rng.integers(1, 257, size=2000).tolist()         # a long queue of requests
    rs, rc = static_batching(lengths, 32), continuous_batching(lengths, 32)
    print(f"2,000 queued requests, 32 slots: static {rs[2]:.0%}, continuous {rc[2]:.0%}")
    assert rc[2] > 0.95 and rc[2] > rs[2]
    for name, u in [("static", rs[2]), ("continuous", rc[2])]:
        c, _ = cost_per_million(3.0, 100, 32)
        print(f"  {name:10s}: ${c / u:.2f} per million useful tokens (same GPU, same model)")
    print("Check 5 passed.")

[TODO] static_batching is not implemented yet: skipping this check.


## Part 6. Speculative decoding: guess, then verify

A small **draft** model with distribution $q$ proposes $\gamma$ tokens, one after another. The large **target** model with distribution $p$ scores all of them in **one** parallel pass. Then, for each drafted token $x$ in order:

* accept $x$ with probability $\min\!\big(1, \; p(x) / q(x)\big)$;
* on the first rejection, sample a replacement from the **residual** $\;\dfrac{\max(0,\, p - q)}{\sum \max(0,\, p - q)}$ and stop;
* if all $\gamma$ are accepted, sample one bonus token from the target.

This rule (Leviathan et al. 2023; Chen et al. 2023) makes every emitted token follow $p$ exactly, so speculative decoding changes the speed but not the output distribution. Write the rule for one drafted token.

In [24]:
# TODO 6: the accept/reject rule for one drafted token.
def accept_or_resample(p, q, x, rng):
    """p, q: numpy probability vectors over the vocabulary (target, draft). x: the token the draft sampled from q.
    rng: a numpy Generator. Return (accepted, token):
      with probability min(1, p[x] / q[x]) return (True, x);
      otherwise return (False, y) with y sampled from max(0, p - q), renormalized."""
    raise NotImplementedError

In [25]:
# Check 6a: one step. Draft from q, apply the rule, and compare the histogram of emitted tokens with p.
p = np.array([0.30, 0.25, 0.20, 0.15, 0.07, 0.03])
q = np.array([0.10, 0.35, 0.05, 0.30, 0.15, 0.05])
rng = np.random.default_rng(9)
N1 = 60_000
first = attempt(accept_or_resample, p, q, 0, rng)
if first is not None:
    out, n_acc = np.zeros(len(p)), 0
    drafts = rng.choice(len(q), size=N1, p=q)
    for x in drafts:
        acc, tok = accept_or_resample(p, q, x, rng)
        out[tok] += 1; n_acc += acc
    emp = out / N1
    tv = 0.5 * np.abs(emp - p).sum()
    print("target p      :", np.round(p, 3))
    print("emitted tokens:", np.round(emp, 3), f" (total variation from p: {tv:.4f})")
    print("draft q       :", np.round(q, 3), f" (total variation from p: {0.5*np.abs(q-p).sum():.4f})")
    print(f"acceptance rate {n_acc/N1:.3f} vs theory sum_x min(p, q) = {np.minimum(p, q).sum():.3f}")
    assert tv < 0.015 and abs(n_acc / N1 - np.minimum(p, q).sum()) < 0.01
    print("Check 6a passed.")

[TODO] accept_or_resample is not implemented yet: skipping this check.


### Full speculative decoding on two toy "language models"

For a clean experiment the target and the draft are **bigram models**: a table `P[prev, next]` and `Q[prev, next]` over a 6-token vocabulary. This keeps every probability known exactly. `speculative_generate` drafts $\gamma$ tokens from `Q`, gets the target's rows for all of them at once (one target call), and applies your rule.

In [26]:
def speculative_generate(P, Q, start, n_tokens, gamma, rng, max_steps=None):
    """Generate n_tokens after `start`. Returns (tokens, target_calls)."""
    seq, calls = [start], 0
    while (n_tokens is None or len(seq) - 1 < n_tokens) and (max_steps is None or calls < max_steps):
        ctx, drafts = seq[-1], []
        for _ in range(gamma):                                    # draft gamma tokens, one by one
            x = rng.choice(Q.shape[1], p=Q[ctx]); drafts.append(x); ctx = x
        ctxs = [seq[-1]] + drafts                                 # one target pass scores every position
        calls += 1
        for i, x in enumerate(drafts):
            acc, tok = accept_or_resample(P[ctxs[i]], Q[ctxs[i]], x, rng)
            seq.append(tok)
            if not acc:
                break
        else:
            seq.append(rng.choice(P.shape[1], p=P[drafts[-1]]))  # all accepted: one bonus token
    return (seq[1:] if n_tokens is None else seq[1:1 + n_tokens]), calls

Vb = 6
rng = np.random.default_rng(10)
P_big = rng.dirichlet(np.full(Vb, 2.0), size=Vb)                          # target bigram table
Q_draft = 0.5 * P_big + 0.5 * rng.dirichlet(np.full(Vb, 2.0), size=Vb)   # a related but different draft

In [27]:
# Check 6b: the distribution of 3-token continuations equals the target's, exactly.
if first is not None:
    from itertools import product
    exact = {s: P_big[0, s[0]] * P_big[s[0], s[1]] * P_big[s[1], s[2]] for s in product(range(Vb), repeat=3)}
    N3 = 20_000

    def tv_of(samples):
        counts = {}
        for s in samples:
            counts[s] = counts.get(s, 0) + 1
        return 0.5 * sum(abs(counts.get(s, 0) / len(samples) - pr) for s, pr in exact.items())

    rng = np.random.default_rng(11)
    spec = [tuple(speculative_generate(P_big, Q_draft, 0, 3, 4, rng)[0]) for _ in range(N3)]

    def plain(T):  # ordinary ancestral sampling from a bigram table
        a = rng.choice(Vb, p=T[0]); b = rng.choice(Vb, p=T[a]); return (a, b, rng.choice(Vb, p=T[b]))
    from_target = [plain(P_big) for _ in range(N3)]
    from_draft = [plain(Q_draft) for _ in range(N3)]

    tv_spec, tv_tgt, tv_drf = tv_of(spec), tv_of(from_target), tv_of(from_draft)
    print(f"total variation from the exact target distribution over {len(exact)} continuations, {N3:,} samples each:")
    print(f"  speculative decoding : {tv_spec:.3f}")
    print(f"  sampling the target  : {tv_tgt:.3f}   (the sampling-noise floor)")
    print(f"  sampling the draft   : {tv_drf:.3f}")
    assert tv_spec < 1.5 * tv_tgt and tv_drf > 2 * tv_tgt
    print("Check 6b passed: speculative output is as close to the target as the target's own samples.")
else:
    print("[TODO] this check needs accept_or_resample: skipping.")

[TODO] this check needs accept_or_resample: skipping.


### When does it pay? Tokens per target step

If each drafted token is accepted with the same probability $\alpha$, one target call emits on average

$$\mathbb{E}[\text{tokens per step}] = \frac{1-\alpha^{\gamma+1}}{1-\alpha}.$$

To test the formula we need a draft with a **constant** acceptance rate. Add a "junk" token that the target never produces, and let the draft put mass $1-\alpha$ on it: $q = \alpha\,p + (1-\alpha)\,\delta_{\text{junk}}$. Then $\sum_x \min(p, q) = \alpha$ in every context. The cell compares the measured tokens per step with the formula for the slides' values ($\gamma = 4$).

The formula is an upper bound on the speedup: it ignores the draft's own cost. Leviathan et al. also give the wall-time improvement $\frac{1-\alpha^{\gamma+1}}{(1-\alpha)(\gamma c + 1)}$, where $c$ is the draft's cost relative to one target call; the slides' 7B/70B pair gives $c \approx 0.1$.

In [28]:
def expected_tokens_per_step(alpha, gamma):
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)

for a, expect in [(0.5, 1.94), (0.8, 3.36), (0.9, 4.10)]:
    assert round(expected_tokens_per_step(a, 4), 2) == expect

if first is not None:
    Vj = Vb + 1                                    # token Vb is "junk"
    P_j = np.zeros((Vj, Vj)); P_j[:Vb, :Vb] = P_big; P_j[Vb, :Vb] = 1 / Vb
    print(f"{'alpha':>5s} {'measured tokens/step':>21s} {'formula':>8s} {'wall-time gain, c = 0.1':>24s}")
    for a in [0.5, 0.8, 0.9]:
        Q_j = a * P_j.copy(); Q_j[:, Vb] += 1 - a
        rng = np.random.default_rng(12)
        toks, calls = speculative_generate(P_j, Q_j, 0, None, 4, rng, max_steps=3000)
        meas, form = len(toks) / calls, expected_tokens_per_step(a, 4)
        assert Vb not in toks                      # junk is never emitted
        print(f"{a:>5.1f} {meas:>21.2f} {form:>8.2f} {form / (4 * 0.1 + 1):>24.2f}")
        assert abs(meas - form) < 0.1
    print("Check 6c passed: measured tokens per step match the formula.")
else:
    print("[TODO] this check needs accept_or_resample: skipping.")

[TODO] this check needs accept_or_resample: skipping.


## Summary

| Part | Trick | What this notebook showed | Changes outputs? |
|---|---|---|---|
| 1 | (budget) | KV cache $= 2LD\times$bytes per token; the cache, not the weights, sets how many users fit; decode speed $\lesssim$ bandwidth / model bytes | - |
| 2 | KV cache | same tokens and logits as recomputing, much less work per step | no |
| 2 | Chunked prefill, prefix reuse | identical cache and logits | no |
| 2 | GQA | cache shrinks by $H/H_{kv}$ | yes (a different model) |
| 3 | Absmax quantization | error roughly doubles per bit removed; per-block scales tame outliers | yes, slightly or more |
| 3 | Pruning | unstructured zeros keep the dense cost; structured pruning shrinks it | yes |
| 3 | Distillation | temperature exposes which wrong answers are nearly right | yes |
| 4 | MoE | compute per token is $k/E$ of the experts; a big batch touches nearly all of them | yes (a different model) |
| 5 | Continuous batching | idle slots vanish while requests wait | no |
| 6 | Speculative decoding | output distribution equals the target's; tokens per step match $\frac{1-\alpha^{\gamma+1}}{1-\alpha}$ | no |

Measure first (TTFT, TPOT, throughput, utilization), find the limiting phase and resource, then change the cheapest thing. Tricks that leave outputs unchanged can ship after a speed test; tricks that change outputs need a quality evaluation.

**Credits.** Inference arithmetic after Pope et al., "Efficiently Scaling Transformer Inference" (MLSys 2023) and Miao et al., "Towards Efficient Generative LLM Serving: A Survey" (ACM CSUR 2025). Llama-2-70B configuration from Touvron et al. (2023). GQA: Ainslie et al. (EMNLP 2023). Absmax and block-wise quantization: Dettmers et al., LLM.int8() (NeurIPS 2022) and QLoRA (NeurIPS 2023). Pruning: Han et al., Deep Compression (ICLR 2016). Distillation: Hinton, Vinyals and Dean (2015). Mixture-of-experts: Shazeer et al. (ICLR 2017); load-balancing loss from Fedus, Zoph and Shazeer, Switch Transformers (JMLR 2022). Continuous batching: Yu et al., Orca (OSDI 2022). Chunked prefill: Agrawal et al., Sarathi-Serve (OSDI 2024). Prefix caching: Zheng et al., SGLang (2024); Kwon et al., vLLM (SOSP 2023). Speculative decoding: Leviathan, Kalman and Matias (ICML 2023); Chen et al., "Accelerating Large Language Model Decoding with Speculative Sampling" (2023).

In [29]:
print(f"Total notebook runtime: {time.time() - T_START:.0f} s")

Total notebook runtime: 1 s
